In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import pickle

print("1. Data Load & VADER Labels...")
# Dhyan rahe: Apni CSV file ka naam sahi ho
df = pd.read_csv('twitter_dataset.csv', encoding='latin-1') 
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

def get_sentiment_label(text):
    compound = sia.polarity_scores(str(text))['compound']
    if compound >= 0.05: return 'Positive'
    elif compound <= -0.05: return 'Negative'
    else: return 'Neutral'

df['Sentiment'] = df['Text'].apply(get_sentiment_label)

# 🌟 Binary Classification: Sirf Positive aur Negative rakh rahe hain
df = df[df['Sentiment'].isin(['Positive', 'Negative'])]

print("2. Smart Cleaning...")
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))
negative_words = ['not', 'no', 'nor', 'isn', "isn't", 'aren', "aren't", 'wasn', "wasn't", 
                  'weren', "weren't", 'hasn', "hasn't", 'haven', "haven't", 'hadn', "hadn't", 
                  'doesn', "doesn't", 'don', "don't", 'didn', "didn't", 'won', "won't", 
                  'wouldn', "wouldn't", 'shan', "shan't", 'shouldn', "shouldn't", 
                  'can', "can't", 'couldn', "couldn't", 'mustn', "mustn't", 'never']

for word in negative_words:
    if word in stop_words:
        stop_words.remove(word)

def smart_clean_tweet(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text) 
    text = text.lower()
    
    # 🔥 NLP PRO TRICK: Negation Handling
    # Ye "not good" ko "not_good" aur "didnt like" ko "didnt_like" bana dega
    text = re.sub(r'\b(not|no|never|isnt|wasnt|arent|didnt|doesnt|dont|cant|couldnt|wouldnt|wont)\s+(\w+)', r'\1_\2', text)
    
    words = text.split()
    cleaned_words = [word for word in words if word not in stop_words]
    return ' '.join(cleaned_words)

df['Smart_Cleaned'] = df['Text'].apply(smart_clean_tweet)
df = df.dropna(subset=['Smart_Cleaned', 'Sentiment'])

print("3. Model Training...")
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_features = vectorizer.fit_transform(df['Smart_Cleaned'])

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_features, df['Sentiment'])

print("4. Saving Files...")
with open('sentiment_model.pkl', 'wb') as file:
    pickle.dump(model, file)
with open('tfidf_vectorizer.pkl', 'wb') as file:
    pickle.dump(vectorizer, file)

print("✅ SUCCESS: Naya SMART Model aur Vectorizer save ho gaya!")

1. Data Load & VADER Labels...
2. Smart Cleaning...
3. Model Training...
4. Saving Files...
✅ SUCCESS: Naya SMART Model aur Vectorizer save ho gaya!
